In [ ]:
import heapq
import random
from bisect import bisect_left, insort
from collections import Counter, defaultdict, deque
from math import gcd

# Greedy, Intervals, Math and Design

---
# 1) Introduction

The remaining interview families. Each is short to code but easy to get wrong without the
key observation, so every *Idea* below states the observation explicitly.

| Family | Key habit |
|---|---|
| Greedy | state the exchange argument: "swapping my choice for any other choice cannot make things better" |
| Intervals | sort by start (merging) or by end (scheduling / fewest removals) |
| Math | look for an invariant or a closed form before simulating |
| Bit manipulation | `x & (x−1)` drops the lowest set bit, `x & −x` isolates it, `a ^ a = 0` |
| Design | pick one structure per required operation, then make them point at each other |

**How to use each question block.** Read the *Problem*, try for five minutes, then read the
*Signal*, then the *Idea*. Tiers: ★★★ Blind 75, ★★ NeetCode 150 / Grind 169 / Top Interview
150, ★ extended.

---
# 2) Greedy

A greedy algorithm commits to the locally best choice and never revisits it. It is correct
only when you can argue that some optimal solution makes the same first choice. If you
cannot find that argument in a minute, the problem is probably DP.

## 2.1 Reachability and scans

### 2.1.1 [55. Jump Game](https://leetcode.com/problems/jump-game/) — ★★★ · Medium

**Problem.** `nums[i]` is the max jump from `i`; can you reach the last index from index 0?  
**Signal.** Track the farthest reachable index.  
**Idea.** Walk left to right; if `i` exceeds `reach`, index `i` is unreachable → false. Otherwise `reach = max(reach, i + nums[i])`.  
**Cost.** O(n) time · O(1) space

In [ ]:
def can_jump(nums: list[int]) -> bool:
    """True if the last index is reachable."""
    reach = 0
    for i, step in enumerate(nums):
        if i > reach:
            return False
        reach = max(reach, i + step)
    return True


assert can_jump([2, 3, 1, 1, 4])
assert not can_jump([3, 2, 1, 0, 4])
print(can_jump([2, 3, 1, 1, 4]))

### 2.1.2 [45. Jump Game II](https://leetcode.com/problems/jump-game-ii/) — ★★ · Medium

**Problem.** Minimum number of jumps to reach the last index (always reachable).  
**Signal.** Implicit BFS by levels: each jump extends the window of reachable indices.  
**Idea.** `end` marks the edge of the current jump's range; `far` is the farthest reachable from inside it. When `i` reaches `end`, take a jump and set `end = far`.  
**Cost.** O(n) time · O(1) space

In [ ]:
def jump(nums: list[int]) -> int:
    """Fewest jumps to reach the end."""
    jumps = end = far = 0
    for i in range(len(nums) - 1):
        far = max(far, i + nums[i])
        if i == end:
            jumps += 1
            end = far
    return jumps


assert jump([2, 3, 1, 1, 4]) == 2
assert jump([2, 3, 0, 1, 4]) == 2
assert jump([0]) == 0
print(jump([2, 3, 1, 1, 4]))

### 2.1.3 [134. Gas Station](https://leetcode.com/problems/gas-station/) — ★★ · Medium

**Problem.** Circular route; `gas[i]` fuel at station `i`, `cost[i]` to drive to the next. Starting index for a full loop, or −1.  
**Signal.** If total gas ≥ total cost a start exists; a failed run rules out every start inside it.  
**Idea.** Walk once keeping `tank`. When it goes negative at `i`, no station between the current start and `i` can work (each would arrive at `i` with even less fuel), so restart at `i + 1`.  
**Cost.** O(n) time · O(1) space

In [ ]:
def can_complete_circuit(gas: list[int], cost: list[int]) -> int:
    """Start index for a full loop, or -1."""
    if sum(gas) < sum(cost):
        return -1
    start = tank = 0
    for i, (g, c) in enumerate(zip(gas, cost)):
        tank += g - c
        if tank < 0:
            start, tank = i + 1, 0
    return start


assert can_complete_circuit([1, 2, 3, 4, 5], [3, 4, 5, 1, 2]) == 3
assert can_complete_circuit([2, 3, 4], [3, 4, 3]) == -1
print(can_complete_circuit([1, 2, 3, 4, 5], [3, 4, 5, 1, 2]))

### 2.1.4 [763. Partition Labels](https://leetcode.com/problems/partition-labels/) — ★★ · Medium

**Problem.** Split `s` into as many parts as possible so each letter appears in at most one part; return part sizes.  
**Signal.** A part must extend to the last occurrence of every letter inside it.  
**Idea.** Record each letter's last index. Scan, stretching `end` to the last occurrence of every letter seen; when `i == end`, close the part.  
**Cost.** O(n) time · O(26) space

In [ ]:
def partition_labels(s: str) -> list[int]:
    """Sizes of maximal-count partitions."""
    last = {ch: i for i, ch in enumerate(s)}
    out, start, end = [], 0, 0
    for i, ch in enumerate(s):
        end = max(end, last[ch])
        if i == end:
            out.append(end - start + 1)
            start = i + 1
    return out


assert partition_labels("ababcbacadefegdehijhklij") == [9, 7, 8]
assert partition_labels("eccbbbbdec") == [10]
print(partition_labels("ababcbacadefegdehijhklij"))

### 2.1.5 [678. Valid Parenthesis String](https://leetcode.com/problems/valid-parenthesis-string/) — ★★ · Medium

**Problem.** `(`, `)` and `*` (which can be `(`, `)` or empty); is the string valid?  
**Signal.** Track the *range* of possible open counts.  
**Idea.** Keep `lo` and `hi`, the min and max possible open brackets. `(` raises both, `)` lowers both, `*` lowers `lo` and raises `hi`. If `hi < 0` it is invalid; clamp `lo` at 0. Valid iff `lo == 0` at the end.  
**Cost.** O(n) time · O(1) space

In [ ]:
def check_valid_string(s: str) -> bool:
    """True if '*' can be assigned to balance s."""
    lo = hi = 0
    for ch in s:
        lo += 1 if ch == "(" else -1
        hi += -1 if ch == ")" else 1
        if hi < 0:
            return False
        lo = max(lo, 0)
    return lo == 0


assert check_valid_string("()")
assert check_valid_string("(*)")
assert check_valid_string("(*))")
assert not check_valid_string("(((*)")
print(check_valid_string("(*))"))

### 2.1.6 [846. Hand of Straights](https://leetcode.com/problems/hand-of-straights/) — ★★ · Medium

**Problem.** Can the cards be split into groups of `group_size` consecutive values?  
**Signal.** The smallest remaining card must start a group.  
**Idea.** Count cards. Repeatedly take the smallest value `x` with a positive count; the group `x, x+1, …, x+size−1` must exist; decrement each by the count of `x` at once.  
**Cost.** O(n log n)

In [ ]:
def is_n_straight_hand(hand: list[int], group_size: int) -> bool:
    """True if hand splits into consecutive groups."""
    if len(hand) % group_size:
        return False
    count = Counter(hand)
    for x in sorted(count):
        need = count[x]
        if need:
            for v in range(x, x + group_size):
                if count[v] < need:
                    return False
                count[v] -= need
    return True


assert is_n_straight_hand([1, 2, 3, 6, 2, 3, 4, 7, 8], 3)
assert not is_n_straight_hand([1, 2, 3, 4, 5], 4)
print(is_n_straight_hand([1, 2, 3, 6, 2, 3, 4, 7, 8], 3))

### 2.1.7 [1899. Merge Triplets to Form Target Triplet](https://leetcode.com/problems/merge-triplets-to-form-target-triplet/) — ★★ · Medium

**Problem.** Merging two triplets takes the element-wise max. Can you obtain `target`?  
**Signal.** Only triplets that never exceed the target in any coordinate are usable.  
**Idea.** Discard any triplet with a component larger than the target (merging it would overshoot forever). Among the rest, check that each target coordinate is matched by some triplet.  
**Cost.** O(n)

In [ ]:
def merge_triplets(triplets: list[list[int]], target: list[int]) -> bool:
    """True if target is reachable by max-merges."""
    hit = set()
    for t in triplets:
        if all(a <= b for a, b in zip(t, target)):
            hit.update(i for i in range(3) if t[i] == target[i])
    return len(hit) == 3


assert merge_triplets([[2, 5, 3], [1, 8, 4], [1, 7, 5]], [2, 7, 5])
assert not merge_triplets([[3, 4, 5], [4, 5, 6]], [3, 2, 5])
print(merge_triplets([[2, 5, 3], [1, 8, 4], [1, 7, 5]], [2, 7, 5]))

### 2.1.8 [135. Candy](https://leetcode.com/problems/candy/) — ★★ · Hard

**Problem.** Each child gets ≥ 1 candy; a child with a higher rating than a neighbour gets more than that neighbour. Minimum total?  
**Signal.** Two passes: satisfy left neighbours, then right neighbours.  
**Idea.** Left-to-right: if rating rises, `candy[i] = candy[i−1] + 1`. Right-to-left: if rating rises toward the left, `candy[i] = max(candy[i], candy[i+1] + 1)`. Each pass fixes one side without breaking the other.  
**Cost.** O(n) time · O(n) space

In [ ]:
def candy(ratings: list[int]) -> int:
    """Minimum candies satisfying neighbour constraints."""
    n = len(ratings)
    out = [1] * n
    for i in range(1, n):
        if ratings[i] > ratings[i - 1]:
            out[i] = out[i - 1] + 1
    for i in range(n - 2, -1, -1):
        if ratings[i] > ratings[i + 1]:
            out[i] = max(out[i], out[i + 1] + 1)
    return sum(out)


assert candy([1, 0, 2]) == 5
assert candy([1, 2, 2]) == 4
print(candy([1, 0, 2]))

### 2.1.9 [1029. Two City Scheduling](https://leetcode.com/problems/two-city-scheduling/) — ★ · Medium

**Problem.** `2n` people, `costs[i] = [a, b]`; send exactly `n` to each city at minimum cost.  
**Signal.** Sort by the *saving* of choosing city A over B.  
**Idea.** Sort by `a − b`. The first `n` people (largest relative benefit from A) go to A, the rest to B.  
**Cost.** O(n log n)

In [ ]:
def two_city_sched_cost(costs: list[list[int]]) -> int:
    """Cheapest balanced assignment."""
    costs = sorted(costs, key=lambda c: c[0] - c[1])
    n = len(costs) // 2
    return sum(a for a, _ in costs[:n]) + sum(b for _, b in costs[n:])


assert two_city_sched_cost([[10, 20], [30, 200], [400, 50], [30, 20]]) == 110
print(two_city_sched_cost([[10, 20], [30, 200], [400, 50], [30, 20]]))

### 2.1.10 [455. Assign Cookies](https://leetcode.com/problems/assign-cookies/) — ★ · Easy

**Problem.** Child `i` is content with a cookie of size ≥ `g[i]`. Maximise content children.  
**Signal.** Sort both; give each child the smallest cookie that satisfies them.  
**Idea.** Two pointers over sorted greed and sizes; a cookie too small for the least greedy remaining child is useless and skipped.  
**Cost.** O(n log n)

In [ ]:
def find_content_children(g: list[int], s: list[int]) -> int:
    """Number of satisfied children."""
    g, s = sorted(g), sorted(s)
    child = 0
    for size in s:
        if child < len(g) and size >= g[child]:
            child += 1
    return child


assert find_content_children([1, 2, 3], [1, 1]) == 1
assert find_content_children([1, 2], [1, 2, 3]) == 2
print(find_content_children([1, 2, 3], [1, 1]))

### 2.1.11 [605. Can Place Flowers](https://leetcode.com/problems/can-place-flowers/) — ★ · Easy

**Problem.** Flowerbed of 0/1, no adjacent flowers; can `n` new flowers be planted?  
**Signal.** Plant greedily at every legal empty spot from the left.  
**Idea.** A 0 whose neighbours (or array edges) are both 0 can be planted; planting as early as possible never blocks more spots than skipping.  
**Cost.** O(n)

In [ ]:
def can_place_flowers(flowerbed: list[int], n: int) -> bool:
    """True if n flowers fit without adjacency."""
    bed = [0] + flowerbed + [0]
    for i in range(1, len(bed) - 1):
        if bed[i - 1] == bed[i] == bed[i + 1] == 0:
            bed[i] = 1
            n -= 1
    return n <= 0


assert can_place_flowers([1, 0, 0, 0, 1], 1)
assert not can_place_flowers([1, 0, 0, 0, 1], 2)
print(can_place_flowers([0, 0, 1, 0, 0], 2))

### 2.1.12 [860. Lemonade Change](https://leetcode.com/problems/lemonade-change/) — ★ · Easy

**Problem.** Lemonade costs 5; customers pay 5, 10 or 20 in order. Can you always give change?  
**Signal.** Simulate, preferring to give a 10 as change for a 20 (keep 5s, which are more flexible).  
**Idea.** Track counts of 5s and 10s. For 20, use `10 + 5` if possible, else `5 + 5 + 5`.  
**Cost.** O(n)

In [ ]:
def lemonade_change(bills: list[int]) -> bool:
    """True if every customer gets correct change."""
    fives = tens = 0
    for b in bills:
        if b == 5:
            fives += 1
        elif b == 10:
            fives, tens = fives - 1, tens + 1
        elif tens:
            fives, tens = fives - 1, tens - 1
        else:
            fives -= 3
        if fives < 0:
            return False
    return True


assert lemonade_change([5, 5, 5, 10, 20])
assert not lemonade_change([5, 5, 10, 10, 20])
print(lemonade_change([5, 5, 5, 10, 20]))

### 2.1.13 [406. Queue Reconstruction by Height](https://leetcode.com/problems/queue-reconstruction-by-height/) — ★ · Medium

**Problem.** People `[h, k]`: `k` people in front are at least as tall. Reconstruct the queue.  
**Signal.** Insert tallest first; shorter people do not affect taller people's `k`.  
**Idea.** Sort by height descending, `k` ascending. Insert each person at index `k`: everyone already placed is at least as tall, so exactly `k` of them end up in front.  
**Cost.** O(n²)

In [ ]:
def reconstruct_queue(people: list[list[int]]) -> list[list[int]]:
    """Queue consistent with every [h, k]."""
    out: list[list[int]] = []
    for p in sorted(people, key=lambda p: (-p[0], p[1])):
        out.insert(p[1], p)
    return out


people = [[7, 0], [4, 4], [7, 1], [5, 0], [6, 1], [5, 2]]
assert reconstruct_queue(people) == [[5, 0], [7, 0], [5, 2], [6, 1], [4, 4], [7, 1]]
print(reconstruct_queue(people))

### 2.1.14 [330. Patching Array](https://leetcode.com/problems/patching-array/) — ★ · Hard

**Problem.** Sorted array; fewest numbers to add so every value in `1..n` is a sum of some subset.  
**Signal.** If all values in `[1, miss)` are coverable, adding `miss` extends coverage to `[1, 2·miss)`.  
**Idea.** Keep `miss`, the smallest uncoverable value. If the next array number is ≤ `miss`, it extends coverage to `miss + num`; otherwise patch with `miss` itself (the largest number that keeps coverage contiguous) and double `miss`.  
**Cost.** O(m + log n)

In [ ]:
def min_patches(nums: list[int], n: int) -> int:
    """Fewest additions covering 1..n."""
    miss, i, patches = 1, 0, 0
    while miss <= n:
        if i < len(nums) and nums[i] <= miss:
            miss += nums[i]
            i += 1
        else:
            miss += miss
            patches += 1
    return patches


assert min_patches([1, 3], 6) == 1
assert min_patches([1, 5, 10], 20) == 2
assert min_patches([1, 2, 2], 5) == 0
print(min_patches([1, 5, 10], 20))

---
# 3) Intervals

| Goal | Sort by | Why |
|---|---|---|
| merge / insert / union | start | overlapping intervals become adjacent |
| fewest removals, most non-overlapping, arrows | **end** | finishing earliest leaves the most room |
| max simultaneous overlap (rooms) | separate starts and ends, or a min-heap of ends | count active intervals at each moment |

Two intervals `[a, b]` and `[c, d]` overlap iff `a <= d and c <= b`.

## 3.1 Merging and scheduling

### 3.1.1 [56. Merge Intervals](https://leetcode.com/problems/merge-intervals/) — ★★★ · Medium

**Problem.** Merge all overlapping intervals.  
**Signal.** Sort by start; overlapping intervals are then consecutive.  
**Idea.** For each interval, if it starts within the last merged interval, extend that interval's end; otherwise start a new one.  
**Cost.** O(n log n)

In [ ]:
def merge_intervals(intervals: list[list[int]]) -> list[list[int]]:
    """Union of overlapping intervals."""
    out: list[list[int]] = []
    for start, end in sorted(intervals):
        if out and start <= out[-1][1]:
            out[-1][1] = max(out[-1][1], end)
        else:
            out.append([start, end])
    return out


assert merge_intervals([[1, 3], [2, 6], [8, 10], [15, 18]]) == [
    [1, 6],
    [8, 10],
    [15, 18],
]
assert merge_intervals([[1, 4], [4, 5]]) == [[1, 5]]
print(merge_intervals([[1, 3], [2, 6], [8, 10], [15, 18]]))

### 3.1.2 [57. Insert Interval](https://leetcode.com/problems/insert-interval/) — ★★★ · Medium

**Problem.** Insert a new interval into sorted, non-overlapping intervals and merge.  
**Signal.** Three phases: before, overlapping, after.  
**Idea.** Copy intervals ending before the new one starts. Merge all intervals overlapping it by widening `[min start, max end]`. Copy the rest.  
**Cost.** O(n)

In [ ]:
def insert_interval(intervals: list[list[int]], new: list[int]) -> list[list[int]]:
    """Intervals with new inserted and merged."""
    out, i, n = [], 0, len(intervals)
    start, end = new
    while i < n and intervals[i][1] < start:
        out.append(intervals[i])
        i += 1
    while i < n and intervals[i][0] <= end:
        start = min(start, intervals[i][0])
        end = max(end, intervals[i][1])
        i += 1
    out.append([start, end])
    return out + intervals[i:]


assert insert_interval([[1, 3], [6, 9]], [2, 5]) == [[1, 5], [6, 9]]
assert insert_interval([[1, 2], [3, 5], [6, 7], [8, 10], [12, 16]], [4, 8]) == [
    [1, 2],
    [3, 10],
    [12, 16],
]
assert insert_interval([], [5, 7]) == [[5, 7]]
print(insert_interval([[1, 3], [6, 9]], [2, 5]))

### 3.1.3 [435. Non-overlapping Intervals](https://leetcode.com/problems/non-overlapping-intervals/) — ★★★ · Medium

**Problem.** Fewest intervals to remove so the rest do not overlap.  
**Signal.** Keep the maximum number of non-overlapping intervals → sort by end (activity selection).  
**Idea.** Choosing the interval that ends earliest leaves the most room for the rest. Count kept intervals; removals = `n − kept`. Touching endpoints do not overlap.  
**Cost.** O(n log n)

In [ ]:
def erase_overlap_intervals(intervals: list[list[int]]) -> int:
    """Minimum removals to eliminate overlaps."""
    kept, last_end = 0, float("-inf")
    for start, end in sorted(intervals, key=lambda iv: iv[1]):
        if start >= last_end:
            kept += 1
            last_end = end
    return len(intervals) - kept


assert erase_overlap_intervals([[1, 2], [2, 3], [3, 4], [1, 3]]) == 1
assert erase_overlap_intervals([[1, 2], [1, 2], [1, 2]]) == 2
assert erase_overlap_intervals([[1, 2], [2, 3]]) == 0
print(erase_overlap_intervals([[1, 2], [2, 3], [3, 4], [1, 3]]))

### 3.1.4 [452. Minimum Number of Arrows to Burst Balloons](https://leetcode.com/problems/minimum-number-of-arrows-to-burst-balloons/) — ★★ · Medium

**Problem.** Balloons are intervals on the x-axis; an arrow at `x` bursts all containing `x`. Fewest arrows?  
**Signal.** Same as activity selection: shoot at the earliest end.  
**Idea.** Sort by end. Shoot at the first balloon's end; it bursts every balloon starting at or before it. The next arrow goes at the end of the first balloon it missed.  
**Cost.** O(n log n)

In [ ]:
def find_min_arrow_shots(points: list[list[int]]) -> int:
    """Fewest arrows to burst all balloons."""
    arrows, arrow_x = 0, float("-inf")
    for start, end in sorted(points, key=lambda p: p[1]):
        if start > arrow_x:
            arrows += 1
            arrow_x = end
    return arrows


assert find_min_arrow_shots([[10, 16], [2, 8], [1, 6], [7, 12]]) == 2
assert find_min_arrow_shots([[1, 2], [3, 4], [5, 6], [7, 8]]) == 4
assert find_min_arrow_shots([[1, 2], [2, 3], [3, 4], [4, 5]]) == 2
print(find_min_arrow_shots([[10, 16], [2, 8], [1, 6], [7, 12]]))

### 3.1.5 [252. Meeting Rooms](https://leetcode.com/problems/meeting-rooms/) — ★★★ · Easy · Premium

**Problem.** Can one person attend all meetings?  
**Signal.** Sort by start and check adjacent overlaps.  
**Idea.** After sorting, any overlap shows up between neighbours.  
**Cost.** O(n log n)

In [ ]:
def can_attend_meetings(intervals: list[list[int]]) -> bool:
    """True if no meetings overlap."""
    intervals = sorted(intervals)
    return all(a[1] <= b[0] for a, b in zip(intervals, intervals[1:]))


assert not can_attend_meetings([[0, 30], [5, 10], [15, 20]])
assert can_attend_meetings([[7, 10], [2, 4]])
print(can_attend_meetings([[7, 10], [2, 4]]))

### 3.1.6 [253. Meeting Rooms II](https://leetcode.com/problems/meeting-rooms-ii/) — ★★★ · Medium · Premium

**Problem.** Minimum number of meeting rooms needed.  
**Signal.** Max number of simultaneously active meetings → min-heap of end times.  
**Idea.** Sort by start. Before placing a meeting, free the room whose meeting ended earliest if it ended by now (pop the heap). Push this meeting's end. The heap size peak is the answer. (Equivalent: sweep sorted starts and ends with two pointers.)  
**Cost.** O(n log n)

In [ ]:
def min_meeting_rooms(intervals: list[list[int]]) -> int:
    """Rooms needed for all meetings."""
    ends: list[int] = []
    for start, end in sorted(intervals):
        if ends and ends[0] <= start:
            heapq.heapreplace(ends, end)
        else:
            heapq.heappush(ends, end)
    return len(ends)


assert min_meeting_rooms([[0, 30], [5, 10], [15, 20]]) == 2
assert min_meeting_rooms([[7, 10], [2, 4]]) == 1
print(min_meeting_rooms([[0, 30], [5, 10], [15, 20]]))

### 3.1.7 [1851. Minimum Interval to Include Each Query](https://leetcode.com/problems/minimum-interval-to-include-each-query/) — ★★ · Hard

**Problem.** For each query point, the size of the smallest interval containing it (−1 if none).  
**Signal.** Offline: sort queries and intervals, sweep with a min-heap keyed by size.  
**Idea.** Process queries in increasing order. Push every interval whose start ≤ query as `(size, end)`. Pop heap tops whose end < query (they can never contain later queries either). The top is the smallest interval containing the query.  
**Cost.** O((n + q) log n)

In [ ]:
def min_interval(intervals: list[list[int]], queries: list[int]) -> list[int]:
    """Smallest containing interval size per query."""
    intervals = sorted(intervals)
    heap: list[tuple[int, int]] = []
    answers: dict[int, int] = {}
    i = 0
    for q in sorted(set(queries)):
        while i < len(intervals) and intervals[i][0] <= q:
            lo, hi = intervals[i]
            heapq.heappush(heap, (hi - lo + 1, hi))
            i += 1
        while heap and heap[0][1] < q:
            heapq.heappop(heap)
        answers[q] = heap[0][0] if heap else -1
    return [answers[q] for q in queries]


assert min_interval([[1, 4], [2, 4], [3, 6], [4, 4]], [2, 3, 4, 5]) == [3, 3, 1, 4]
assert min_interval([[2, 3], [2, 5], [1, 8], [20, 25]], [2, 19, 5, 22]) == [2, -1, 4, 6]
print(min_interval([[1, 4], [2, 4], [3, 6], [4, 4]], [2, 3, 4, 5]))

### 3.1.8 [228. Summary Ranges](https://leetcode.com/problems/summary-ranges/) — ★★ · Easy

**Problem.** Sorted unique integers → list of ranges like `"0->2"`.  
**Signal.** Group consecutive runs.  
**Idea.** Extend a run while the next number is one larger; emit `"a->b"` or `"a"`.  
**Cost.** O(n)

In [ ]:
def summary_ranges(nums: list[int]) -> list[str]:
    """Compact range strings."""
    out, i = [], 0
    while i < len(nums):
        j = i
        while j + 1 < len(nums) and nums[j + 1] == nums[j] + 1:
            j += 1
        out.append(str(nums[i]) if i == j else f"{nums[i]}->{nums[j]}")
        i = j + 1
    return out


assert summary_ranges([0, 1, 2, 4, 5, 7]) == ["0->2", "4->5", "7"]
assert summary_ranges([0, 2, 3, 4, 6, 8, 9]) == ["0", "2->4", "6", "8->9"]
print(summary_ranges([0, 1, 2, 4, 5, 7]))

### 3.1.9 [986. Interval List Intersections](https://leetcode.com/problems/interval-list-intersections/) — ★ · Medium

**Problem.** Intersections of two sorted, disjoint interval lists.  
**Signal.** Two pointers; advance the interval that ends first.  
**Idea.** The overlap of the current pair is `[max(starts), min(ends)]` if non-empty. The interval ending first cannot intersect anything further in the other list, so advance it.  
**Cost.** O(m + n)

In [ ]:
def interval_intersection(a: list[list[int]], b: list[list[int]]) -> list[list[int]]:
    """Pairwise intersections of two interval lists."""
    out, i, j = [], 0, 0
    while i < len(a) and j < len(b):
        lo = max(a[i][0], b[j][0])
        hi = min(a[i][1], b[j][1])
        if lo <= hi:
            out.append([lo, hi])
        if a[i][1] < b[j][1]:
            i += 1
        else:
            j += 1
    return out


a = [[0, 2], [5, 10], [13, 23], [24, 25]]
b = [[1, 5], [8, 12], [15, 24], [25, 26]]
assert interval_intersection(a, b) == [
    [1, 2],
    [5, 5],
    [8, 10],
    [15, 23],
    [24, 24],
    [25, 25],
]
print(interval_intersection(a, b))

### 3.1.10 [759. Employee Free Time](https://leetcode.com/problems/employee-free-time/) — ★ · Hard · Premium

**Problem.** Each employee has sorted, disjoint working intervals; return the common free time.  
**Signal.** Merge all intervals, then report the gaps.  
**Idea.** Flatten and sort every interval by start, merge on the fly, and whenever the next start exceeds the merged end, the gap `[end, next_start]` is free for everyone.  
**Cost.** O(N log N)

In [ ]:
def employee_free_time(schedule: list[list[list[int]]]) -> list[list[int]]:
    """Gaps where nobody works."""
    intervals = sorted(iv for emp in schedule for iv in emp)
    out, end = [], intervals[0][1]
    for s, e in intervals[1:]:
        if s > end:
            out.append([end, s])
        end = max(end, e)
    return out


assert employee_free_time([[[1, 2], [5, 6]], [[1, 3]], [[4, 10]]]) == [[3, 4]]
assert employee_free_time([[[1, 3], [6, 7]], [[2, 4]], [[2, 5], [9, 12]]]) == [
    [5, 6],
    [7, 9],
]
print(employee_free_time([[[1, 2], [5, 6]], [[1, 3]], [[4, 10]]]))

### 3.1.11 [1229. Meeting Scheduler](https://leetcode.com/problems/meeting-scheduler/) — ★ · Medium · Premium

**Problem.** Two people's free slots; the earliest common slot of length `duration`, or `[]`.  
**Signal.** Interval List Intersections, stopping at the first overlap long enough.  
**Idea.** Sort both lists. Two pointers; the overlap `[max starts, min ends]` works if its length ≥ `duration`. Advance the slot ending first.  
**Cost.** O(m log m + n log n)

In [ ]:
def min_available_duration(
    slots1: list[list[int]], slots2: list[list[int]], duration: int
) -> list[int]:
    """Earliest common slot of the required length."""
    slots1, slots2 = sorted(slots1), sorted(slots2)
    i = j = 0
    while i < len(slots1) and j < len(slots2):
        lo = max(slots1[i][0], slots2[j][0])
        hi = min(slots1[i][1], slots2[j][1])
        if hi - lo >= duration:
            return [lo, lo + duration]
        if slots1[i][1] < slots2[j][1]:
            i += 1
        else:
            j += 1
    return []


s1 = [[10, 50], [60, 120], [140, 210]]
s2 = [[0, 15], [60, 70]]
assert min_available_duration(s1, s2, 8) == [60, 68]
assert min_available_duration(s1, s2, 12) == []
print(min_available_duration(s1, s2, 8))

### 3.1.12 [729. My Calendar I](https://leetcode.com/problems/my-calendar-i/) — ★ · Medium

**Problem.** `book(start, end)` succeeds if it does not overlap any existing booking.  
**Signal.** Keep bookings sorted; only the neighbours of the insertion point can overlap.  
**Idea.** Store sorted starts with their ends. Binary search where the new start would go; check the previous booking's end and the next booking's start.  
**Cost.** O(log n) search, O(n) insert into a list (O(log n) with a balanced tree)

In [ ]:
class MyCalendar:
    """Non-overlapping booking calendar."""

    def __init__(self) -> None:
        self.bookings: list[tuple[int, int]] = []

    def book(self, start: int, end: int) -> bool:
        i = bisect_left(self.bookings, (start, end))
        if i > 0 and self.bookings[i - 1][1] > start:
            return False
        if i < len(self.bookings) and self.bookings[i][0] < end:
            return False
        insort(self.bookings, (start, end))
        return True


cal = MyCalendar()
assert [cal.book(10, 20), cal.book(15, 25), cal.book(20, 30)] == [True, False, True]
print(cal.bookings)

---
# 4) Math and Geometry

## 4.1 Number manipulation

### 4.1.1 [7. Reverse Integer](https://leetcode.com/problems/reverse-integer/) — ★★ · Medium

**Problem.** Reverse the digits of a 32-bit signed integer; return 0 on overflow.  
**Signal.** Pop digits with `% 10` and `// 10`, check the 32-bit range.  
**Idea.** Work on `abs(x)` and restore the sign. In fixed-width languages check overflow before `rev * 10 + digit`; in Python check the final value.  
**Cost.** O(digits)

In [ ]:
def reverse_integer(x: int) -> int:
    """Digit-reversed x, or 0 on 32-bit overflow."""
    sign = -1 if x < 0 else 1
    x, rev = abs(x), 0
    while x:
        x, digit = divmod(x, 10)
        rev = rev * 10 + digit
    rev *= sign
    return rev if -(2**31) <= rev <= 2**31 - 1 else 0


assert reverse_integer(123) == 321
assert reverse_integer(-123) == -321
assert reverse_integer(120) == 21
assert reverse_integer(1534236469) == 0
print(reverse_integer(-123))

### 4.1.2 [9. Palindrome Number](https://leetcode.com/problems/palindrome-number/) — ★★ · Easy

**Problem.** Is an integer a palindrome, without converting to a string?  
**Signal.** Reverse only half the digits.  
**Idea.** Negative numbers and numbers ending in 0 (except 0) are not palindromes. Move digits from `x` into `rev` until `rev >= x`; then compare `x == rev` or `x == rev // 10` (odd length).  
**Cost.** O(log x)

In [ ]:
def is_palindrome_number(x: int) -> bool:
    """True if x reads the same reversed."""
    if x < 0 or (x % 10 == 0 and x != 0):
        return False
    rev = 0
    while x > rev:
        x, digit = divmod(x, 10)
        rev = rev * 10 + digit
    return x == rev or x == rev // 10


assert is_palindrome_number(121)
assert not is_palindrome_number(-121)
assert not is_palindrome_number(10)
print(is_palindrome_number(12321))

### 4.1.3 [66. Plus One](https://leetcode.com/problems/plus-one/) — ★★ · Easy

**Problem.** Digits array of a number; add one.  
**Signal.** Carry from the end.  
**Idea.** Walk from the last digit: a 9 becomes 0 and carries; any other digit is incremented and we stop. All 9s → prepend a 1.  
**Cost.** O(n)

In [ ]:
def plus_one(digits: list[int]) -> list[int]:
    """Digits of the number plus one."""
    digits = digits.copy()
    for i in range(len(digits) - 1, -1, -1):
        if digits[i] < 9:
            digits[i] += 1
            return digits
        digits[i] = 0
    return [1] + digits


assert plus_one([1, 2, 3]) == [1, 2, 4]
assert plus_one([9, 9]) == [1, 0, 0]
print(plus_one([4, 3, 2, 1]))

### 4.1.4 [202. Happy Number](https://leetcode.com/problems/happy-number/) — ★★ · Easy

**Problem.** Repeatedly replace `n` by the sum of squares of its digits; does it reach 1?  
**Signal.** The sequence either hits 1 or cycles → cycle detection.  
**Idea.** Floyd's slow/fast pointers on the "next" function (or a seen set). If the fast pointer reaches 1, it is happy; if the pointers meet elsewhere, it cycles.  
**Cost.** O(log n) per step, few steps

In [ ]:
def is_happy(n: int) -> bool:
    """True if n is a happy number."""

    def nxt(x: int) -> int:
        return sum(int(d) ** 2 for d in str(x))

    slow, fast = n, nxt(n)
    while fast != 1 and slow != fast:
        slow, fast = nxt(slow), nxt(nxt(fast))
    return fast == 1


assert is_happy(19)
assert not is_happy(2)
print(is_happy(19))

### 4.1.5 [50. Pow(x, n)](https://leetcode.com/problems/powx-n/) — ★★ · Medium

**Problem.** Compute `xⁿ` for integer `n` (possibly negative).  
**Signal.** Fast exponentiation by squaring.  
**Idea.** Use the binary digits of `n`: square the base each step and multiply into the result when the current bit is 1. Negative `n` → compute `(1/x)^{-n}`.  
**Cost.** O(log n)

In [ ]:
def my_pow(x: float, n: int) -> float:
    """x raised to n."""
    if n < 0:
        x, n = 1 / x, -n
    result = 1.0
    while n:
        if n & 1:
            result *= x
        x *= x
        n >>= 1
    return result


assert abs(my_pow(2.0, 10) - 1024.0) < 1e-9
assert abs(my_pow(2.1, 3) - 9.261) < 1e-9
assert abs(my_pow(2.0, -2) - 0.25) < 1e-9
print(my_pow(2.0, 10))

### 4.1.6 [43. Multiply Strings](https://leetcode.com/problems/multiply-strings/) — ★★ · Medium

**Problem.** Multiply two non-negative numbers given as strings, without big-int conversion.  
**Signal.** Grade-school multiplication: digit `i` × digit `j` lands in positions `i + j` and `i + j + 1`.  
**Idea.** Allocate `m + n` result digits. For each pair from the right, add the product to `res[i + j + 1]` and push the carry into `res[i + j]`. Strip leading zeros.  
**Cost.** O(mn)

In [ ]:
def multiply(num1: str, num2: str) -> str:
    """Product of two decimal strings."""
    res = [0] * (len(num1) + len(num2))
    for i in range(len(num1) - 1, -1, -1):
        for j in range(len(num2) - 1, -1, -1):
            total = int(num1[i]) * int(num2[j]) + res[i + j + 1]
            res[i + j + 1] = total % 10
            res[i + j] += total // 10
    out = "".join(map(str, res)).lstrip("0")
    return out or "0"


assert multiply("2", "3") == "6"
assert multiply("123", "456") == "56088"
assert multiply("0", "52") == "0"
print(multiply("123", "456"))

### 4.1.7 [172. Factorial Trailing Zeroes](https://leetcode.com/problems/factorial-trailing-zeroes/) — ★★ · Medium

**Problem.** Number of trailing zeros in `n!`.  
**Signal.** Each trailing zero needs a factor 10 = 2 × 5, and 5s are rarer → count factors of 5.  
**Idea.** `n // 5 + n // 25 + n // 125 + …` (multiples of 25 contribute two 5s, etc.).  
**Cost.** O(log n)

In [ ]:
def trailing_zeroes(n: int) -> int:
    """Trailing zeros of n factorial."""
    count = 0
    while n:
        n //= 5
        count += n
    return count


assert trailing_zeroes(3) == 0
assert trailing_zeroes(5) == 1
assert trailing_zeroes(100) == 24
print(trailing_zeroes(100))

### 4.1.8 [204. Count Primes](https://leetcode.com/problems/count-primes/) — ★ · Medium

**Problem.** Number of primes strictly less than `n`.  
**Signal.** Sieve of Eratosthenes.  
**Idea.** Mark multiples of each prime starting at `p²` (smaller multiples were marked by smaller primes). Only primes up to `√n` need sieving.  
**Cost.** O(n log log n) time · O(n) space

In [ ]:
def count_primes(n: int) -> int:
    """Primes below n."""
    if n < 3:
        return 0
    is_prime = [True] * n
    is_prime[0] = is_prime[1] = False
    for p in range(2, int(n**0.5) + 1):
        if is_prime[p]:
            is_prime[p * p :: p] = [False] * len(range(p * p, n, p))
    return sum(is_prime)


assert count_primes(10) == 4
assert count_primes(0) == 0
assert count_primes(1_000_000) == 78498
print(count_primes(10))

### 4.1.9 [1071. Greatest Common Divisor of Strings](https://leetcode.com/problems/greatest-common-divisor-of-strings/) — ★ · Easy

**Problem.** Largest string `x` that divides both `s` and `t` (each is `x` repeated).  
**Signal.** A common divisor exists iff `s + t == t + s`; its length is `gcd(len(s), len(t))`.  
**Idea.** If the concatenations differ, no divisor exists. Otherwise the prefix of length `gcd(|s|, |t|)` divides both.  
**Cost.** O(m + n)

In [ ]:
def gcd_of_strings(s: str, t: str) -> str:
    """Largest common divisor string."""
    return s[: gcd(len(s), len(t))] if s + t == t + s else ""


assert gcd_of_strings("ABCABC", "ABC") == "ABC"
assert gcd_of_strings("ABABAB", "ABAB") == "AB"
assert gcd_of_strings("LEET", "CODE") == ""
print(gcd_of_strings("ABABAB", "ABAB"))

### 4.1.10 [29. Divide Two Integers](https://leetcode.com/problems/divide-two-integers/) — ★ · Medium

**Problem.** Integer division truncating toward zero without `*`, `/` or `%`; clamp to 32-bit.  
**Signal.** Subtract shifted (doubled) divisors — long division in binary.  
**Idea.** Work with absolute values. For bit `k` from 31 down, if `divisor << k` fits into the remaining dividend, subtract it and add `1 << k` to the quotient.  
**Cost.** O(32)

In [ ]:
def divide(dividend: int, divisor: int) -> int:
    """Truncated quotient within 32-bit range."""
    negative = (dividend < 0) != (divisor < 0)
    a, b = abs(dividend), abs(divisor)
    quotient = 0
    for k in range(31, -1, -1):
        if (b << k) <= a:
            a -= b << k
            quotient |= 1 << k
    quotient = -quotient if negative else quotient
    return max(-(2**31), min(2**31 - 1, quotient))


assert divide(10, 3) == 3
assert divide(7, -3) == -2
assert divide(-(2**31), -1) == 2**31 - 1
print(divide(10, 3))

## 4.2 Geometry and randomness

### 4.2.1 [149. Max Points on a Line](https://leetcode.com/problems/max-points-on-a-line/) — ★★ · Hard

**Problem.** Maximum number of points on one straight line.  
**Signal.** Fix an anchor point, group the others by slope.  
**Idea.** For each anchor, compute the reduced direction `(dy, dx) / gcd` with a canonical sign and count equal directions; the best count + 1 is a candidate. Integer directions avoid floating-point slope errors.  
**Cost.** O(n²)

In [ ]:
def max_points(points: list[list[int]]) -> int:
    """Most points sharing a line."""
    if len(points) < 3:
        return len(points)
    best = 0
    for i, (x1, y1) in enumerate(points):
        slopes: Counter[tuple[int, int]] = Counter()
        for x2, y2 in points[i + 1 :]:
            dx, dy = x2 - x1, y2 - y1
            g = gcd(dx, dy)
            dx, dy = dx // g, dy // g
            if dx < 0 or (dx == 0 and dy < 0):
                dx, dy = -dx, -dy
            slopes[(dx, dy)] += 1
        best = max(best, max(slopes.values(), default=0) + 1)
    return best


assert max_points([[1, 1], [2, 2], [3, 3]]) == 3
assert max_points([[1, 1], [3, 2], [5, 3], [4, 1], [2, 3], [1, 4]]) == 4
print(max_points([[1, 1], [3, 2], [5, 3], [4, 1], [2, 3], [1, 4]]))

### 4.2.2 [2013. Detect Squares](https://leetcode.com/problems/detect-squares/) — ★★ · Medium

**Problem.** `add(point)`; `count(point)` = number of axis-aligned squares with the query point as one corner and three added points as the others (duplicates count separately).  
**Signal.** Choose the diagonal corner; the other two corners are then determined.  
**Idea.** Store a counter of points. For each stored point `(x, y)` that forms a diagonal with the query `(qx, qy)` (`|x − qx| == |y − qy| > 0`), multiply the counts of `(x, y)`, `(qx, y)` and `(x, qy)`.  
**Cost.** O(1) add · O(distinct points) count

In [ ]:
class DetectSquares:
    """Count axis-aligned squares through a query point."""

    def __init__(self) -> None:
        self.points: Counter[tuple[int, int]] = Counter()

    def add(self, point: list[int]) -> None:
        self.points[tuple(point)] += 1

    def count(self, point: list[int]) -> int:
        qx, qy = point
        total = 0
        for (x, y), n in self.points.items():
            if abs(x - qx) == abs(y - qy) and x != qx:
                total += n * self.points[(qx, y)] * self.points[(x, qy)]
        return total


ds = DetectSquares()
for p in [[3, 10], [11, 2], [3, 2]]:
    ds.add(p)
assert ds.count([11, 10]) == 1 and ds.count([14, 8]) == 0
ds.add([11, 2])
assert ds.count([11, 10]) == 2
print("ok")

### 4.2.3 [384. Shuffle an Array](https://leetcode.com/problems/shuffle-an-array/) — ★ · Medium

**Problem.** `shuffle()` returns a uniformly random permutation; `reset()` restores the original.  
**Signal.** Fisher–Yates shuffle.  
**Idea.** For `i` from the end, swap `a[i]` with a uniformly random `a[j]`, `j ≤ i`. Each of the `n!` permutations is produced with probability `1/n!`.  
**Cost.** O(n) per shuffle

In [ ]:
class Shuffler:
    """Uniform random shuffling with reset."""

    def __init__(self, nums: list[int]) -> None:
        self.original = nums.copy()

    def reset(self) -> list[int]:
        return self.original.copy()

    def shuffle(self) -> list[int]:
        a = self.original.copy()
        for i in range(len(a) - 1, 0, -1):
            j = random.randint(0, i)
            a[i], a[j] = a[j], a[i]
        return a


random.seed(0)
sh = Shuffler([1, 2, 3])
counts = Counter(tuple(sh.shuffle()) for _ in range(6000))
assert len(counts) == 6 and min(counts.values()) > 850
assert sh.reset() == [1, 2, 3]
print(counts)

---
# 5) Bit Manipulation

| Trick | Meaning |
|---|---|
| `x & 1` | lowest bit (odd?) |
| `x & (x − 1)` | clear the lowest set bit (power of two ⇔ result is 0) |
| `x & −x` | isolate the lowest set bit |
| `a ^ a = 0`, `a ^ 0 = a` | XOR cancels pairs |
| `x >> k & 1` | k-th bit |

Python ints are unbounded, so emulate 32-bit behaviour with a mask `0xFFFFFFFF` where the
problem relies on overflow.

## 5.1 XOR and bit counting

### 5.1.1 [136. Single Number](https://leetcode.com/problems/single-number/) — ★★ · Easy

**Problem.** Every element appears twice except one; find it in O(1) space.  
**Signal.** XOR cancels pairs.  
**Idea.** XOR everything; pairs vanish and the single number remains.  
**Cost.** O(n) time · O(1) space

In [ ]:
def single_number(nums: list[int]) -> int:
    """The element without a pair."""
    out = 0
    for x in nums:
        out ^= x
    return out


assert single_number([2, 2, 1]) == 1
assert single_number([4, 1, 2, 1, 2]) == 4
print(single_number([4, 1, 2, 1, 2]))

### 5.1.2 [137. Single Number II](https://leetcode.com/problems/single-number-ii/) — ★★ · Medium

**Problem.** Every element appears three times except one; find it in O(1) space.  
**Signal.** Count each bit modulo 3.  
**Idea.** For each of the 32 bits, sum that bit across all numbers; a remainder of 1 mod 3 means the single number has that bit. Convert the result back from 32-bit two's complement.  
**Cost.** O(32 n)

In [ ]:
def single_number_ii(nums: list[int]) -> int:
    """The element appearing once among triples."""
    out = 0
    for b in range(32):
        if sum((x >> b) & 1 for x in nums) % 3:
            out |= 1 << b
    return out - (1 << 32) if out >= 1 << 31 else out


assert single_number_ii([2, 2, 3, 2]) == 3
assert single_number_ii([0, 1, 0, 1, 0, 1, 99]) == 99
assert single_number_ii([-2, -2, 1, 1, 4, 1, 4, 4, -4, -2]) == -4
print(single_number_ii([0, 1, 0, 1, 0, 1, 99]))

### 5.1.3 [260. Single Number III](https://leetcode.com/problems/single-number-iii/) — ★ · Medium

**Problem.** Exactly two elements appear once, all others twice; find both.  
**Signal.** XOR all → `a ^ b`; split numbers by one bit where `a` and `b` differ.  
**Idea.** `diff = a ^ b` is non-zero; its lowest set bit `diff & −diff` separates `a` from `b`. XOR each group separately to recover each number.  
**Cost.** O(n) time · O(1) space

In [ ]:
def single_number_iii(nums: list[int]) -> list[int]:
    """The two unpaired elements."""
    diff = 0
    for x in nums:
        diff ^= x
    low = diff & -diff
    a = 0
    for x in nums:
        if x & low:
            a ^= x
    return sorted([a, diff ^ a])


assert single_number_iii([1, 2, 1, 3, 2, 5]) == [3, 5]
assert single_number_iii([-1, 0]) == [-1, 0]
print(single_number_iii([1, 2, 1, 3, 2, 5]))

### 5.1.4 [268. Missing Number](https://leetcode.com/problems/missing-number/) — ★★★ · Easy

**Problem.** `n` distinct numbers from `0..n`; find the missing one.  
**Signal.** XOR indices and values (or use the sum formula).  
**Idea.** XOR of `0..n` and all values leaves the missing number; equivalently `n(n+1)/2 − sum(nums)`.  
**Cost.** O(n) time · O(1) space

In [ ]:
def missing_number(nums: list[int]) -> int:
    """The value in 0..n not present."""
    out = len(nums)
    for i, x in enumerate(nums):
        out ^= i ^ x
    return out


assert missing_number([3, 0, 1]) == 2
assert missing_number([0, 1]) == 2
assert missing_number([9, 6, 4, 2, 3, 5, 7, 0, 1]) == 8
print(missing_number([3, 0, 1]))

### 5.1.5 [191. Number of 1 Bits](https://leetcode.com/problems/number-of-1-bits/) — ★★★ · Easy

**Problem.** Count set bits (Hamming weight).  
**Signal.** `n & (n − 1)` drops the lowest set bit.  
**Idea.** Repeat `n &= n − 1` until zero; the number of iterations is the number of set bits.  
**Cost.** O(set bits)

In [ ]:
def hamming_weight(n: int) -> int:
    """Number of 1 bits."""
    count = 0
    while n:
        n &= n - 1
        count += 1
    return count


assert hamming_weight(11) == 3
assert hamming_weight(128) == 1
assert hamming_weight(2147483645) == 30
print(hamming_weight(11))

### 5.1.6 [338. Counting Bits](https://leetcode.com/problems/counting-bits/) — ★★★ · Easy

**Problem.** For every `i` in `0..n`, the number of set bits.  
**Signal.** Reuse smaller answers: `bits[i] = bits[i >> 1] + (i & 1)`.  
**Idea.** Dropping the last bit gives a smaller number already computed; add back the last bit.  
**Cost.** O(n)

In [ ]:
def count_bits(n: int) -> list[int]:
    """Set-bit counts for 0..n."""
    bits = [0] * (n + 1)
    for i in range(1, n + 1):
        bits[i] = bits[i >> 1] + (i & 1)
    return bits


assert count_bits(2) == [0, 1, 1]
assert count_bits(5) == [0, 1, 1, 2, 1, 2]
print(count_bits(5))

### 5.1.7 [190. Reverse Bits](https://leetcode.com/problems/reverse-bits/) — ★★★ · Easy

**Problem.** Reverse the bits of a 32-bit unsigned integer.  
**Signal.** Shift bits out of `n` and into the result, 32 times.  
**Idea.** Each step: `out = (out << 1) | (n & 1)`, `n >>= 1`.  
**Cost.** O(32)

In [ ]:
def reverse_bits(n: int) -> int:
    """32-bit reversal of n."""
    out = 0
    for _ in range(32):
        out = (out << 1) | (n & 1)
        n >>= 1
    return out


assert reverse_bits(0b00000010100101000001111010011100) == 964176192
assert reverse_bits(0b11111111111111111111111111111101) == 3221225471
print(reverse_bits(43261596))

### 5.1.8 [371. Sum of Two Integers](https://leetcode.com/problems/sum-of-two-integers/) — ★★★ · Medium

**Problem.** Add two integers without `+` or `−`.  
**Signal.** XOR adds without carry; AND shifted left is the carry.  
**Idea.** Repeat `a, b = a ^ b, (a & b) << 1` until no carry. In Python, mask to 32 bits so negative numbers terminate, then convert back from two's complement.  
**Cost.** O(32)

In [ ]:
def get_sum(a: int, b: int) -> int:
    """a + b using only bit operations."""
    mask = 0xFFFFFFFF
    while b & mask:
        a, b = (a ^ b) & mask, ((a & b) << 1) & mask
    a &= mask
    return a if a < 1 << 31 else a - (1 << 32)


assert get_sum(1, 2) == 3
assert get_sum(2, 3) == 5
assert get_sum(-1, 1) == 0
assert get_sum(-12, -8) == -20
print(get_sum(-12, -8))

### 5.1.9 [67. Add Binary](https://leetcode.com/problems/add-binary/) — ★★ · Easy

**Problem.** Sum of two binary strings as a binary string.  
**Signal.** Digit-by-digit addition with carry from the right.  
**Idea.** Walk both strings from the end, adding bits and the carry; emit `total % 2`, carry `total // 2`.  
**Cost.** O(max(m, n))

In [ ]:
def add_binary(a: str, b: str) -> str:
    """Binary sum of two binary strings."""
    i, j, carry = len(a) - 1, len(b) - 1, 0
    out = []
    while i >= 0 or j >= 0 or carry:
        total = carry
        if i >= 0:
            total += int(a[i])
            i -= 1
        if j >= 0:
            total += int(b[j])
            j -= 1
        out.append(str(total % 2))
        carry = total // 2
    return "".join(reversed(out))


assert add_binary("11", "1") == "100"
assert add_binary("1010", "1011") == "10101"
print(add_binary("1010", "1011"))

### 5.1.10 [201. Bitwise AND of Numbers Range](https://leetcode.com/problems/bitwise-and-of-numbers-range/) — ★★ · Medium

**Problem.** Bitwise AND of all integers in `[left, right]`.  
**Signal.** The result is the common binary prefix of `left` and `right`.  
**Idea.** Any bit below the common prefix flips somewhere in the range, so it ANDs to 0. Shift both right until equal, then shift back.  
**Cost.** O(32)

In [ ]:
def range_bitwise_and(left: int, right: int) -> int:
    """AND of all numbers in [left, right]."""
    shift = 0
    while left != right:
        left >>= 1
        right >>= 1
        shift += 1
    return left << shift


assert range_bitwise_and(5, 7) == 4
assert range_bitwise_and(0, 0) == 0
assert range_bitwise_and(1, 2147483647) == 0
print(range_bitwise_and(5, 7))

### 5.1.11 [231. Power of Two](https://leetcode.com/problems/power-of-two/) — ★ · Easy

**Problem.** Is `n` a power of two?  
**Signal.** A power of two has exactly one set bit.  
**Idea.** `n > 0 and n & (n − 1) == 0`.  
**Cost.** O(1)

In [ ]:
def is_power_of_two(n: int) -> bool:
    """True if n is 2**k."""
    return n > 0 and n & (n - 1) == 0


assert is_power_of_two(1) and is_power_of_two(16)
assert not is_power_of_two(3) and not is_power_of_two(0)
print(is_power_of_two(1024))

### 5.1.12 [461. Hamming Distance](https://leetcode.com/problems/hamming-distance/) — ★ · Easy

**Problem.** Number of positions where the bits of `x` and `y` differ.  
**Signal.** XOR marks differing bits; count them.  
**Idea.** `bin(x ^ y).count("1")` (or `hamming_weight(x ^ y)`).  
**Cost.** O(32)

In [ ]:
def hamming_distance(x: int, y: int) -> int:
    """Differing bit count."""
    return hamming_weight(x ^ y)


assert hamming_distance(1, 4) == 2
assert hamming_distance(3, 1) == 1
print(hamming_distance(1, 4))

### 5.1.13 [1318. Minimum Flips to Make a OR b Equal to c](https://leetcode.com/problems/minimum-flips-to-make-a-or-b-equal-to-c/) — ★ · Medium

**Problem.** Fewest bit flips in `a` and `b` so that `a | b == c`.  
**Signal.** Decide bit by bit.  
**Idea.** If the `c` bit is 1, we need at least one of `a`, `b` bits set: 1 flip if both are 0. If the `c` bit is 0, both must be 0: flip each set one.  
**Cost.** O(32)

In [ ]:
def min_flips(a: int, b: int, c: int) -> int:
    """Bit flips making a | b == c."""
    flips = 0
    while a or b or c:
        x, y, z = a & 1, b & 1, c & 1
        flips += (x + y) if z == 0 else int(x == y == 0)
        a, b, c = a >> 1, b >> 1, c >> 1
    return flips


assert min_flips(2, 6, 5) == 3
assert min_flips(4, 2, 7) == 1
assert min_flips(1, 2, 3) == 0
print(min_flips(2, 6, 5))

---
# 6) Design

Design questions grade whether each operation hits its required complexity and whether the
class is easy to read. Write the operation → required cost table first, then pick structures:
hash map for O(1) lookup, array for O(1) random access, deque/linked list for O(1) ends,
heap for O(log n) min/max, sorted list + bisect for order queries.

## 6.1 Data-structure design

### 6.1.1 [380. Insert Delete GetRandom O(1)](https://leetcode.com/problems/insert-delete-getrandom-o1/) — ★★ · Medium

**Problem.** `insert`, `remove` and `get_random` (uniform), all average O(1).  
**Signal.** Array for random access + hash map value → index; delete by swapping with the last element.  
**Idea.** To remove `val`, move the last element into `val`'s slot, update its index in the map, then pop the array's end. `random.choice` on the array is uniform.  
**Cost.** O(1) average per op

In [ ]:
class RandomizedSet:
    """Set with O(1) insert, remove and uniform sampling."""

    def __init__(self) -> None:
        self.values: list[int] = []
        self.index: dict[int, int] = {}

    def insert(self, val: int) -> bool:
        if val in self.index:
            return False
        self.index[val] = len(self.values)
        self.values.append(val)
        return True

    def remove(self, val: int) -> bool:
        if val not in self.index:
            return False
        i, last = self.index[val], self.values[-1]
        self.values[i], self.index[last] = last, i
        self.values.pop()
        del self.index[val]
        return True

    def get_random(self) -> int:
        return random.choice(self.values)


rs = RandomizedSet()
assert rs.insert(1) and not rs.remove(2) and rs.insert(2)
assert rs.get_random() in (1, 2)
assert rs.remove(1) and not rs.insert(2) and rs.get_random() == 2
print(rs.values)

### 6.1.2 [362. Design Hit Counter](https://leetcode.com/problems/design-hit-counter/) — ★★ · Medium · Premium

**Problem.** `hit(timestamp)` and `get_hits(timestamp)` = hits in the past 300 seconds; timestamps are non-decreasing.  
**Signal.** Queue of timestamps; evict entries older than the window.  
**Idea.** Append each hit. Before counting, pop from the front while the oldest timestamp is `<= timestamp − 300`. (For heavy traffic, store `(timestamp, count)` pairs or a circular array of 300 buckets.)  
**Cost.** amortised O(1)

In [ ]:
class HitCounter:
    """Hits within the last 300 seconds."""

    def __init__(self) -> None:
        self.hits: deque[int] = deque()

    def hit(self, timestamp: int) -> None:
        self.hits.append(timestamp)

    def get_hits(self, timestamp: int) -> int:
        while self.hits and self.hits[0] <= timestamp - 300:
            self.hits.popleft()
        return len(self.hits)


hc = HitCounter()
for t in (1, 2, 3):
    hc.hit(t)
assert hc.get_hits(4) == 3
hc.hit(300)
assert hc.get_hits(300) == 4 and hc.get_hits(301) == 3
print(hc.get_hits(301))

### 6.1.3 [346. Moving Average from Data Stream](https://leetcode.com/problems/moving-average-from-data-stream/) — ★ · Easy · Premium

**Problem.** Average of the last `size` values of a stream.  
**Signal.** Fixed window → deque with a running sum.  
**Idea.** Append the value and add it to the sum; if the window is too large, pop the oldest and subtract it.  
**Cost.** O(1) per value

In [ ]:
class MovingAverage:
    """Average over a sliding window."""

    def __init__(self, size: int) -> None:
        self.size = size
        self.window: deque[int] = deque()
        self.total = 0

    def next(self, val: int) -> float:
        self.window.append(val)
        self.total += val
        if len(self.window) > self.size:
            self.total -= self.window.popleft()
        return self.total / len(self.window)


ma = MovingAverage(3)
out = [ma.next(v) for v in [1, 10, 3, 5]]
assert out == [1.0, 5.5, 14 / 3, 6.0]
print(out)

### 6.1.4 [622. Design Circular Queue](https://leetcode.com/problems/design-circular-queue/) — ★ · Medium

**Problem.** Fixed-capacity FIFO queue on an array with `en_queue`, `de_queue`, `front`, `rear`.  
**Signal.** Ring buffer: head index + size, positions taken modulo capacity.  
**Idea.** The tail position is `(head + size) % capacity`. Enqueue writes there; dequeue advances `head`. Track `size` to distinguish full from empty.  
**Cost.** O(1) per op

In [ ]:
class MyCircularQueue:
    """Ring-buffer queue."""

    def __init__(self, k: int) -> None:
        self.data = [0] * k
        self.head = self.size = 0

    def en_queue(self, value: int) -> bool:
        if self.size == len(self.data):
            return False
        self.data[(self.head + self.size) % len(self.data)] = value
        self.size += 1
        return True

    def de_queue(self) -> bool:
        if not self.size:
            return False
        self.head = (self.head + 1) % len(self.data)
        self.size -= 1
        return True

    def front(self) -> int:
        return self.data[self.head] if self.size else -1

    def rear(self) -> int:
        return (
            self.data[(self.head + self.size - 1) % len(self.data)] if self.size else -1
        )


q = MyCircularQueue(3)
assert [q.en_queue(v) for v in (1, 2, 3, 4)] == [True, True, True, False]
assert q.rear() == 3 and q.de_queue() and q.en_queue(4) and q.rear() == 4
assert q.front() == 2
print(q.data)

### 6.1.5 [706. Design HashMap](https://leetcode.com/problems/design-hashmap/) — ★ · Easy

**Problem.** Hash map with `put`, `get`, `remove` without built-in hash tables.  
**Signal.** Array of buckets (separate chaining).  
**Idea.** Bucket index = `key % size` (a prime size spreads keys). Each bucket is a list of `[key, value]` pairs scanned linearly; load stays low so operations are O(1) on average.  
**Cost.** O(1) average

In [ ]:
class MyHashMap:
    """Separate-chaining hash map."""

    def __init__(self, size: int = 1009) -> None:
        self.buckets: list[list[list[int]]] = [[] for _ in range(size)]

    def _bucket(self, key: int) -> list[list[int]]:
        return self.buckets[key % len(self.buckets)]

    def put(self, key: int, value: int) -> None:
        bucket = self._bucket(key)
        for pair in bucket:
            if pair[0] == key:
                pair[1] = value
                return
        bucket.append([key, value])

    def get(self, key: int) -> int:
        for k, v in self._bucket(key):
            if k == key:
                return v
        return -1

    def remove(self, key: int) -> None:
        bucket = self._bucket(key)
        bucket[:] = [pair for pair in bucket if pair[0] != key]


hm = MyHashMap()
hm.put(1, 1)
hm.put(2, 2)
assert hm.get(1) == 1 and hm.get(3) == -1
hm.put(2, 1)
assert hm.get(2) == 1
hm.remove(2)
assert hm.get(2) == -1
print("ok")

### 6.1.6 [1570. Dot Product of Two Sparse Vectors](https://leetcode.com/problems/dot-product-of-two-sparse-vectors/) — ★ · Medium · Premium

**Problem.** Compute dot products of sparse vectors efficiently.  
**Signal.** Store only non-zero entries as `index → value`.  
**Idea.** Iterate over the smaller dictionary and look up matching indices in the other.  
**Cost.** O(min(nnz₁, nnz₂))

In [ ]:
class SparseVector:
    """Sparse vector keyed by non-zero index."""

    def __init__(self, nums: list[int]) -> None:
        self.entries = {i: x for i, x in enumerate(nums) if x}

    def dot_product(self, other: "SparseVector") -> int:
        small, big = sorted((self.entries, other.entries), key=len)
        return sum(x * big.get(i, 0) for i, x in small.items())


v1, v2 = SparseVector([1, 0, 0, 2, 3]), SparseVector([0, 3, 0, 4, 0])
assert v1.dot_product(v2) == 8
assert (
    SparseVector([0, 1, 0, 0, 2, 0, 0]).dot_product(SparseVector([1, 0, 0, 0, 3, 0, 4]))
    == 6
)
print(v1.dot_product(v2))

### 6.1.7 [284. Peeking Iterator](https://leetcode.com/problems/peeking-iterator/) — ★ · Medium

**Problem.** Wrap an iterator to support `peek()` in addition to `next()` and `has_next()`.  
**Signal.** Cache one element ahead.  
**Idea.** Pre-fetch the next element into `self._next`; `peek` returns it, `next` returns it and pre-fetches again.  
**Cost.** O(1) per op

In [ ]:
class PeekingIterator:
    """Iterator with one-element lookahead."""

    _END = object()

    def __init__(self, iterator) -> None:
        self.it = iterator
        self._next = next(self.it, self._END)

    def peek(self):
        return self._next

    def next(self):
        value = self._next
        self._next = next(self.it, self._END)
        return value

    def has_next(self) -> bool:
        return self._next is not self._END


pi = PeekingIterator(iter([1, 2, 3]))
assert pi.next() == 1 and pi.peek() == 2 and pi.next() == 2 and pi.next() == 3
assert not pi.has_next()
print("ok")

### 6.1.8 [341. Flatten Nested List Iterator](https://leetcode.com/problems/flatten-nested-list-iterator/) — ★ · Medium

**Problem.** Iterate over the integers of an arbitrarily nested list.  
**Signal.** Stack of iterators (lazy) — or flatten eagerly with recursion.  
**Idea.** Keep a stack of list iterators. `has_next` advances until the top iterator yields an integer (stash it) or the stack empties; nested lists push a new iterator.  
**Cost.** amortised O(1) per element

In [ ]:
class NestedIterator:
    """Lazy flattening iterator."""

    def __init__(self, nested: list) -> None:
        self.stack = [iter(nested)]
        self.buffer: list[int] = []

    def has_next(self) -> bool:
        while not self.buffer and self.stack:
            item = next(self.stack[-1], None)
            if item is None:
                self.stack.pop()
            elif isinstance(item, list):
                self.stack.append(iter(item))
            else:
                self.buffer.append(item)
        return bool(self.buffer)

    def next(self) -> int:
        self.has_next()
        return self.buffer.pop()


it = NestedIterator([[1, 1], 2, [1, [4, [6]]], []])
out = []
while it.has_next():
    out.append(it.next())
assert out == [1, 1, 2, 1, 4, 6]
print(out)

### 6.1.9 [1396. Design Underground System](https://leetcode.com/problems/design-underground-system/) — ★ · Medium

**Problem.** Check-in/check-out events; `get_average_time(start, end)` over completed trips.  
**Signal.** Two hash maps: open trips by customer, and totals by `(start, end)`.  
**Idea.** Check-in stores `(station, time)` for the customer. Check-out pops it and adds the trip duration to `totals[(start, end)] = [sum, count]`. The average is `sum / count`.  
**Cost.** O(1) per op

In [ ]:
class UndergroundSystem:
    """Average trip times between stations."""

    def __init__(self) -> None:
        self.open: dict[int, tuple[str, int]] = {}
        self.totals: dict[tuple[str, str], list[int]] = defaultdict(lambda: [0, 0])

    def check_in(self, cid: int, station: str, t: int) -> None:
        self.open[cid] = (station, t)

    def check_out(self, cid: int, station: str, t: int) -> None:
        start, t0 = self.open.pop(cid)
        total = self.totals[(start, station)]
        total[0] += t - t0
        total[1] += 1

    def get_average_time(self, start: str, end: str) -> float:
        total, count = self.totals[(start, end)]
        return total / count


u = UndergroundSystem()
u.check_in(45, "Leyton", 3)
u.check_in(32, "Paradise", 8)
u.check_out(45, "Waterloo", 15)
u.check_out(32, "Cambridge", 22)
assert u.get_average_time("Paradise", "Cambridge") == 14.0
u.check_in(10, "Leyton", 24)
u.check_out(10, "Waterloo", 38)
assert u.get_average_time("Leyton", "Waterloo") == 13.0
print(u.get_average_time("Leyton", "Waterloo"))

### 6.1.10 [588. Design In-Memory File System](https://leetcode.com/problems/design-in-memory-file-system/) — ★ · Hard · Premium

**Problem.** `ls`, `mkdir`, `add_content_to_file` and `read_content_from_file` on absolute paths.  
**Signal.** A trie of path components; each node is a directory or a file.  
**Idea.** Each node holds `children` (dict) and `content` (string, `None` for directories). Walk path components creating nodes as needed. `ls` on a file returns its name; on a directory, its sorted children.  
**Cost.** O(path length) per op (plus sorting for `ls`)

In [ ]:
class FileSystem:
    """In-memory hierarchical file system."""

    def __init__(self) -> None:
        self.root: dict = {"children": {}, "content": None}

    def _walk(self, path: str) -> dict:
        node = self.root
        for part in filter(None, path.split("/")):
            node = node["children"].setdefault(part, {"children": {}, "content": None})
        return node

    def ls(self, path: str) -> list[str]:
        node = self._walk(path)
        if node["content"] is not None:
            return [path.split("/")[-1]]
        return sorted(node["children"])

    def mkdir(self, path: str) -> None:
        self._walk(path)

    def add_content_to_file(self, path: str, content: str) -> None:
        node = self._walk(path)
        node["content"] = (node["content"] or "") + content

    def read_content_from_file(self, path: str) -> str:
        return self._walk(path)["content"]


fs = FileSystem()
assert fs.ls("/") == []
fs.mkdir("/a/b/c")
fs.add_content_to_file("/a/b/c/d", "hello")
assert fs.ls("/") == ["a"] and fs.ls("/a/b/c") == ["d"] and fs.ls("/a/b/c/d") == ["d"]
assert fs.read_content_from_file("/a/b/c/d") == "hello"
print("ok")

---
# 7) Conclusion

| Pattern | Recognise it by | Core move | Cost |
|---|---|---|---|
| Farthest reach | jump games | track max reachable index / BFS window | O(n) |
| Restart on failure | gas station, Kadane-like scans | a failed prefix rules out every start inside it | O(n) |
| Last occurrence | partition labels | stretch the window to the last index of every letter | O(n) |
| Range of states | wildcard parentheses | track `[lo, hi]` of possible counts | O(n) |
| Two-pass constraints | candy | fix left neighbours, then right | O(n) |
| Sort by key + greedy | two cities, queue by height, cookies | exchange argument picks the order | O(n log n) |
| Coverage doubling | patching array | add the smallest missing value | O(m + log n) |
| Merge intervals | merge / insert / free time | sort by start, extend the last | O(n log n) |
| Activity selection | removals, arrows | sort by end, keep earliest finishers | O(n log n) |
| Active count | meeting rooms, query intervals | heap of ends | O(n log n) |
| Digit math | reverse, palindrome number, multiply strings | `divmod(x, 10)`, carry arrays | O(digits) |
| Fast power / sieve / gcd | pow, primes, string gcd | squaring, mark multiples from p², `s+t == t+s` | O(log n) / O(n log log n) |
| XOR tricks | single / missing numbers | pairs cancel, split by a differing bit | O(n) |
| Bit DP | counting bits | `bits[i >> 1] + (i & 1)` | O(n) |
| Design | O(1) operations | combine map + array / deque / ring buffer | O(1) |